In [1]:
# Gather links with unstructured
# from unstructured.partition.html import partition_html
# cnn_lite_url = "https://lite.cnn.com/"
# elements = partition_html(url=cnn_lite_url)

# links = []

# for element in elements:
#     if element.metadata.link_urls:
#         relative_link = element.metadata.link_urls[0][1:]
#         if relative_link.startswith("2024"):
#             links.append(f"{cnn_lite_url}{relative_link}")

# print(f"We retrieved {len(links)} links to documents from {cnn_lite_url}")

In [2]:
# Ingest individual articles with Langchain UnstructuredURLLoader
# from langchain.document_loaders import UnstructuredURLLoader
# loaders = UnstructuredURLLoader(urls=links[:200], show_progress_bar=True)

# docs = loaders.load()
# #print(docs[0])
# print(f"We retrieved {len(docs)} links to documents from {cnn_lite_url}")

# # Variable docs is a list of langchain_core.documents.base.Document -> convert it to a list of strings
# docs_as_list_of_strings = [docs[i].page_content for i in range(len(docs))]

In [3]:
# For reproducibility we can save and load the documents in a file
# -------------------------------------------------------
# Save the list of documents as a JSON file
# import json
# with open("docs_as_list_of_strings.json", "w") as file:
#     json.dump(docs_as_list_of_strings, file)

# Load the list of documents from a JSON file
import json
with open("docs_as_list_of_strings.json", "r") as file:
    docs_as_list_of_strings = json.load(file)
# -------------------------------------------------------

# Each document of the list is a very long text, so we split each document into smaller chuncks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

# Two-dimensional list where each sublist represent a document splitted in chunks
docs_as_list_of_chuncks = [] 
for item in docs_as_list_of_strings:
    docs_as_list_of_chuncks.append(text_splitter.split_text(item))

# Load ibm-granite/granite-guardian-hap-38m model
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_name_or_path = 'ibm-granite/granite-guardian-hap-38m'
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

# Calculate HAP probability for every chunk. Save predictions and probabilities for every chunk in a two-dimensional list
prediction_results = []
probability_results = []
for chunks in docs_as_list_of_chuncks:
    input = tokenizer(chunks, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**input).logits
        prediction_list = torch.argmax(logits, dim=1).detach().numpy().tolist() # Binary prediction where label 1 indicates toxicity.
        prediction_results.append(prediction_list)
        probability_list = torch.softmax(logits, dim=1).detach().numpy()[:,1].tolist() # Probability of toxicity.
        probability_results.append(probability_list)

# Find the indices of the chuncks with HAP prediction of 1
indices = [(i, j) for i, row in enumerate(prediction_results) for j, value in enumerate(row) if value == 1]
# Save the chunks with HAP probability >90% in a key-value dictionary where:
# - key: is the index of the original document
# - value: is a list containg the chunks with HAP content
matrix_of_doc_with_hap = {}
for tup in indices:
    print(tup)  # This prints the tuple
    i=tup[0]; j=tup[1];
    #print(prediction_results[i][j])
    #print(f"Probability: {probability_results[i][j]}, Sentence: {docs_as_list_of_chuncks[i][j]}")
    if(probability_results[i][j] >= 0.90):
        if(i not in matrix_of_doc_with_hap):
            matrix_of_doc_with_hap[i] = []
        matrix_of_doc_with_hap[i].append(docs_as_list_of_chuncks[i][j])
        print(f"Probability: {probability_results[i][j]}, Sentence: {docs_as_list_of_chuncks[i][j]}")
print(f"There are {len(matrix_of_doc_with_hap)} documents with at east one sentence with HAP probability > 90%. {list(matrix_of_doc_with_hap.keys())}.")

/Users/mrinalduzzi/Desktop/Projects/RAG-with-watsonx/HAP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(0, 52)
(0, 94)
(0, 109)
Probability: 0.9138151407241821, Sentence: first set in latex and her now-signature white clown face. When it’s time for “Good Luck, Babe!”
(2, 19)
(3, 59)
Probability: 0.9971562623977661, Sentence: “We were stupid and didn’t take it seriously. We were not responsible; it was a mistake not to
(12, 79)
(19, 24)
(19, 26)
Probability: 0.9303997755050659, Sentence: “I’ve used the term hypocrites because we support peaceful protest, and we facilitate that all the
(21, 175)
(25, 65)
Probability: 0.9272968173027039, Sentence: sections of the Wall.”
(34, 41)
(34, 49)
(38, 50)
Probability: 0.9897632598876953, Sentence: You’re a little f**cking confused.”
(38, 135)
Probability: 0.9396336674690247, Sentence: more than that, the thing that f**ks me up, honestly, is knowing that I don’t know exactly what
(38, 144)
(38, 145)
(38, 147)
(40, 19)
Probability: 0.9032238125801086, Sentence: be pouring from her abdomen.
(40, 27)
(45, 27)
Probability: 0.9545580148696899, Sentence: 

In [4]:
# Helper functions
import nltk
from nltk.tokenize import sent_tokenize, regexp_tokenize

# Ensure you have the required NLTK data
nltk.download('punkt')

def split_text_preserve_newlines(text):
    """
    Split the given text into sentences while preserving newlines.
    1. Splits the text into paragraphs based on double newlines (\n\n).
    2. Tokenizes each paragraph separately into sentences.
    3. Reassembles the sentences while inserting double newlines between paragraphs.
    
    Args:
    text (str): The input text to split.
    
    Returns:
    List[str]: A list of sentences with preserved newlines.
    """
    # Split text by newline characters first
    paragraphs = text.split('\n\n')
    
    # Tokenize each paragraph separately
    sentences_with_newlines = []
    for para in paragraphs:
        sentences = sent_tokenize(para)
        if sentences:
            # Add newline between paragraphs
            sentences_with_newlines.extend(sentences)
            sentences_with_newlines.append('')  # This will add a '\n\n' in the final text
    
    # Remove the last extra newline added
    if sentences_with_newlines and sentences_with_newlines[-1] == '':
        sentences_with_newlines.pop()
    
    return sentences_with_newlines

def rebuild_text_with_newlines(sentences):
    """
    Rebuild the text from a list of sentences with preserved newlines. Joins sentences back into text, inserting double newlines where appropriate.
    
    Args:
    sentences (List[str]): A list of sentences to join.
    
    Returns:
    str: The rebuilt text with preserved newlines.
    """
    rebuilt_text = ''
    for sentence in sentences:
        if sentence == '':
            rebuilt_text += '\n\n'
        else:
            rebuilt_text += sentence + ' '
    
    return rebuilt_text.strip()


def find_substring_indices(full_strings, substrings):
    """
    Find the indices of substrings within a list of full strings.

    Parameters:
    full_strings (list[str]): A list of full strings to search for substrings.
    substrings (list[str]): A list of substrings to search for within the full strings.

    Returns:
    list[int]: A list of indices corresponding to the full strings that contain at least one substring.
    """
    indices = []
    
    # Iterate through each full string
    for i, full_string in enumerate(full_strings):
        # Check if any of the substrings is found in the current full string
        if any(sub in full_string for sub in substrings):
            indices.append(i)
    
    return indices


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mrinalduzzi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
import pandas as pd
import requests

from os import environ
credentials = {
    "url": environ.get("WATSONX_URL"),
    "apikey": environ.get("WATSONX_API_KEY")
}
project_id = environ.get("PRJ_ID")

# Generate Bearer token
url = 'https://iam.cloud.ibm.com/identity/token'
myobj = {'grant_type': 'urn:ibm:params:oauth:grant-type:apikey', 'apikey':credentials["apikey"]}
#use the 'headers' parameter to set the HTTP headers:
x = requests.post(url, data = myobj, headers = {"Content-Type": "application/x-www-form-urlencoded"})
bearer_token = x.json()['access_token']


matrix_of_docs_rewritten = {} # dict (dictionary) that maps integer keys to a single strings (e.g. 38: <document_rewritten>")
for key in list(matrix_of_doc_with_hap.keys()):
    old_doc = docs_as_list_of_strings[key]
    list_of_hap_strings = matrix_of_doc_with_hap[key]

    # --- Retrieve only the sentences with HAP content ---
    # Split the text into sentences
    full_strings = split_text_preserve_newlines(old_doc)

    substrings = list_of_hap_strings

    # Find the indices of strings containing any of the substrings
    matching_indices = find_substring_indices(full_strings, substrings)

    #print("Indices of matching strings:", matching_indices)
    # -----------------------------------------------------

    # -----------------------   GRANITE   ------------------------
    # Build the prompt starting from the template and the instructions

    #Ensure that the revised sentences no longer have harmful or offensive language but retain the original meaning or intention in a more respectful and appropriate way.
    # Initialize matrix_of_docs_rewritten element with the original document
    for i in range(len(matching_indices)):
        prompt_granite = """Rewrite the following sentence to remove any harmful, abusive, or inappropriate content while maintaining the original meaning where possible. Ensure the revised sentence is respectful, neutral, and appropriate for all audiences. The output should contain only the revised sentence itself, without any additional words, explanations, or punctuation marks.

Input: That was a stupid decision and you’re an idiot for making it.
Output: That was a decision that didn’t work out as planned.

Input: Why are you so incompetent? Can’t you do anything right?
Output: Why are you so ineffective? Can’t you do anything right?

Input: It’s absolutely pathetic how you can’t even get the simplest things right, and it’s clear that you’re just a complete f**king failure in every aspect.
Output: It’s disappointing how you can’t even get the simplest things right, and it’s clear that you need some improvements.


Input: """ + full_strings[matching_indices[i]] + """
Output: """

        body_granite = {
            "input": prompt_granite,
            "parameters": {
                "decoding_method": "greedy",
                "max_new_tokens": 200,
                "repetition_penalty": 1.1
            },
            "model_id": "ibm/granite-13b-instruct-v2",
            "project_id": project_id
        }
        #print(body_granite)

        # Make the API call to the LLM model
        authorization = "Bearer " + bearer_token
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Authorization": authorization
        }

        response = requests.post(
            credentials["url"],
            headers=headers,
            json=body_granite
        )

        if response.status_code != 200:
            raise Exception("Non-200 response: " + str(response.text))

        result_granite = response.json()["results"][0]["generated_text"]
        # Remove the \n\n at the end of a string if it exists
        if result_granite.endswith('\n\n'):
            result_granite = result_granite[:-2]
        if result_granite.endswith('  '):
            result_granite = result_granite[:-2]
        
        print(matching_indices[i], "original :", full_strings[matching_indices[i]])
        print(matching_indices[i], "rewritten:", result_granite)
        # Change the original sentence with the revised sentence
        full_strings[matching_indices[i]] = result_granite
        
    matrix_of_docs_rewritten[key] = rebuild_text_with_newlines(full_strings)
    
    # --------------------------------------------------------------

32 original : “I’ve used the term hypocrites because we support peaceful protest, and we facilitate that all the time.
32 rewritten: “We call hypocrites because we support peaceful protest, and we facilitate that all the time.”
66 original : Although the couple managed to have a great time traveling independently without knowing Chinese, they decided to hire a private guide for their visit to the Great Wall “as it facilitates logistics and allows you to go to (lesser-traveled) sections of the Wall.”
66 rewritten: Although the couple managed to have a great time traveling independently without knowing Chinese, they decided to hire a private guide for their visit to the Great Wall “since it facilitates logistics and allows you to go to (lesser-traveled) sections of the wall.”
57 original : You’re a little f**cking confused.”
57 rewritten: You're a little confused."
147 original : “And that’s because of my teammates and trying to put myself in that situation that they described emotionall

In [6]:
# Print one of the revised documents
print(matrix_of_docs_rewritten[38])

CNN 9/11/2024 

What we know about the police detention of Miami Dolphins star Tyreek Hill 

By George Ramsay, Ben Morse, Wayne Sterling, David Close, Kevin Dotson and Homero De la Fuente, CNN 

Updated: 6:54 PM EDT, Tue September 10, 2024 

Source: CNN 

It’s been an eventful few days for Miami Dolphins star Tyreek Hill, who was handcuffed and detained by Miami-Dade Police shortly before his team’s opening game Sunday. 

The detention was just temporary, and Hill later scored a stellar 80-yard touchdown in a close win over the Jacksonville Jaguars. He celebrated the score by putting his arms behind his back with his wrists together – a cheeky nod to his pregame incident. 

But off the field, the detention – as well as police interactions with two other Dolphins players – has led to a public back-and-forth between the NFL team and local police and has renewed the debate over how law enforcement handles traffic stops and interacts with members of the public. 

On Monday night, police re

In [7]:
# Check if there are some differences between the original document and the revised one
import difflib

# Split the texts into lists of lines
lines1 = matrix_of_docs_rewritten[38].splitlines()
lines2 = docs_as_list_of_strings[38].splitlines()

# Use unified_diff to find differences
diff = difflib.unified_diff(lines1, lines2, fromfile='text1', tofile='text2')

# Print the differences
print('\n'.join(diff))

--- text1

+++ text2

@@ -1,143 +1,143 @@

-CNN 9/11/2024 
+CNN 9/11/2024
 
-What we know about the police detention of Miami Dolphins star Tyreek Hill 
+What we know about the police detention of Miami Dolphins star Tyreek Hill
 
-By George Ramsay, Ben Morse, Wayne Sterling, David Close, Kevin Dotson and Homero De la Fuente, CNN 
+By George Ramsay, Ben Morse, Wayne Sterling, David Close, Kevin Dotson and Homero De la Fuente, CNN
 
-Updated: 6:54 PM EDT, Tue September 10, 2024 
+Updated: 6:54 PM EDT, Tue September 10, 2024
 
-Source: CNN 
+Source: CNN
 
-It’s been an eventful few days for Miami Dolphins star Tyreek Hill, who was handcuffed and detained by Miami-Dade Police shortly before his team’s opening game Sunday. 
+It’s been an eventful few days for Miami Dolphins star Tyreek Hill, who was handcuffed and detained by Miami-Dade Police shortly before his team’s opening game Sunday.
 
-The detention was just temporary, and Hill later scored a stellar 80-yard touchdown in a close win